# 01 — VisDrone2019-DET: Download, Conversion, Integrity, Split Freeze

**Project:** RAOD++ · **Phase:** F4 (data preparation) · **Protocol:** v1.0 (frozen)

This notebook:
1. downloads VisDrone2019-DET train+val,
2. converts annotations to YOLO format with an explicit, auditable converter,
3. runs integrity checks (counts, corrupt images, class distribution),
4. **freezes the split** (file lists + SHA256 hashes),
5. runs leakage audit #1 (filename overlap + exact-duplicate MD5 scan),
6. writes the data card to `output/`.

Expected runtime: ~15–25 min (download ~1.6 GB + checks). GPU not required.


## CONFIG — single source of truth (no hard-coded values below this cell)

In [ ]:
from pathlib import Path

CONFIG = {
    # --- roots ---
    "PROJECT_ROOT": Path("/home/ahmet/RAODpp"),   # <-- EDIT to your machine
    # --- dataset ---
    "DATASET_NAME": "VisDrone2019-DET",
    "URLS": {
        "train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
        "val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
    },
    # protocol v1.0 expected counts
    "EXPECTED": {"train_images": 6471, "val_images": 548},
    # VisDrone raw annotation: bbox_left,bbox_top,w,h,score,category,truncation,occlusion
    # categories 1..10 kept (-> 0..9); score==0 rows (ignored regions) and
    # category 0 ("ignored") / 11 ("others") are dropped.
    "CLASSES": ["pedestrian","people","bicycle","car","van","truck",
                "tricycle","awning-tricycle","bus","motor"],
    "SEED": 42,
}

ROOT = CONFIG["PROJECT_ROOT"]
DIRS = {k: ROOT / k for k in ["dataset","model","output","fig","tex","notebooks"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
DATA = DIRS["dataset"] / "visdrone"
DATA.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", ROOT)
for k, d in DIRS.items():
    print(f"  {k:9s} -> {d}  (exists={d.exists()})")


## Environment record (protocol §3: `output/environment.txt`)

In [ ]:
import sys, subprocess, platform, datetime

env_file = DIRS["output"] / "environment.txt"
with open(env_file, "w") as f:
    f.write(f"# RAOD++ environment record — {datetime.datetime.now().isoformat()}\n")
    f.write(f"python={sys.version}\nplatform={platform.platform()}\n\n")
    f.write(subprocess.run([sys.executable,"-m","pip","freeze"],
                           capture_output=True, text=True).stdout)
print("Wrote:", env_file)
print("Line count:", sum(1 for _ in open(env_file)))


## Download train + val archives
Uses `ultralytics.utils.downloads.download` (resumable, checks zip integrity). Only train and val per protocol v1.0 — test-dev is not used.

In [ ]:
from ultralytics.utils.downloads import download

for split, url in CONFIG["URLS"].items():
    download([url], dir=DATA, unzip=True, delete=False)

# verification output
for split in ["train","val"]:
    p = DATA / f"VisDrone2019-DET-{split}"
    n_img = len(list((p/"images").glob("*.jpg")))
    n_ann = len(list((p/"annotations").glob("*.txt")))
    print(f"{split}: images={n_img}  raw_annotations={n_ann}")


## Explicit VisDrone → YOLO conversion
Auditable converter (protocol §4.1): keep `score==1` and `1 <= category <= 10`; class id = category − 1; boxes clipped to image, then normalized to cx,cy,w,h.

In [ ]:
from PIL import Image
from tqdm import tqdm

def convert_split(split):
    base = DATA / f"VisDrone2019-DET-{split}"
    lbl_dir = base / "labels"; lbl_dir.mkdir(exist_ok=True)
    n_boxes = n_dropped = 0
    for ann in tqdm(sorted((base/"annotations").glob("*.txt")), desc=split):
        img_p = base/"images"/(ann.stem + ".jpg")
        W, H = Image.open(img_p).size
        lines = []
        for row in open(ann):
            r = row.strip().split(",")
            if len(r) < 8: continue
            x, y, w, h = map(float, r[:4]); score, cat = int(r[4]), int(r[5])
            if score == 0 or not (1 <= cat <= 10):
                n_dropped += 1; continue
            # clip to image bounds
            x2, y2 = min(x+w, W), min(y+h, H); x, y = max(x,0), max(y,0)
            bw, bh = x2-x, y2-y
            if bw <= 1 or bh <= 1: n_dropped += 1; continue
            cx, cy = (x+bw/2)/W, (y+bh/2)/H
            lines.append(f"{cat-1} {cx:.6f} {cy:.6f} {bw/W:.6f} {bh/H:.6f}")
            n_boxes += 1
        (lbl_dir/(ann.stem+".txt")).write_text("\n".join(lines))
    return n_boxes, n_dropped

for split in ["train","val"]:
    kept, dropped = convert_split(split)
    print(f"{split}: kept_boxes={kept}  dropped(ignored/others/degenerate)={dropped}")


## Integrity check 1 — counts vs protocol, corrupt-image scan

In [ ]:
corrupt = {}
for split in ["train","val"]:
    base = DATA / f"VisDrone2019-DET-{split}"
    imgs = sorted((base/"images").glob("*.jpg"))
    bad = []
    for p in tqdm(imgs, desc=f"verify-{split}"):
        try:
            with Image.open(p) as im: im.verify()
        except Exception:
            bad.append(p.name)
    corrupt[split] = bad
    exp = CONFIG["EXPECTED"][f"{split}_images"]
    status = "OK" if len(imgs) == exp else "MISMATCH"
    print(f"{split}: found={len(imgs)} expected={exp} [{status}]  corrupt={len(bad)}")

assert not corrupt["train"] and not corrupt["val"], f"Corrupt images found: {corrupt}"
print("Corrupt-image scan: CLEAN")


## Integrity check 2 — class distribution (figure saved to `fig/`)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from collections import Counter

counts = {}
for split in ["train","val"]:
    c = Counter()
    for lbl in (DATA/f"VisDrone2019-DET-{split}"/"labels").glob("*.txt"):
        for line in open(lbl):
            if line.strip(): c[int(line.split()[0])] += 1
    counts[split] = c

fig, ax = plt.subplots(figsize=(7,3.2))
x = np.arange(len(CONFIG["CLASSES"])); w = 0.38
ax.bar(x-w/2, [counts["train"].get(i,0) for i in x], w, label="train")
ax.bar(x+w/2, [counts["val"].get(i,0)   for i in x], w, label="val")
ax.set_xticks(x); ax.set_xticklabels(CONFIG["CLASSES"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("instances"); ax.set_yscale("log"); ax.legend(frameon=False)
fig.tight_layout()
for ext in ["pdf","png"]:
    fig.savefig(DIRS["fig"]/f"visdrone_class_dist.{ext}", dpi=300)
plt.show()
print({CONFIG["CLASSES"][i]: counts["train"].get(i,0) for i in x})


## Split freeze — file lists + SHA256 (protocol §1)
After this cell, `train_files.txt` / `val_files.txt` and their hashes are the **only** authoritative split definition. Any later mismatch aborts the run.

In [ ]:
import hashlib, json

hashes = {}
for split in ["train","val"]:
    files = sorted(p.name for p in (DATA/f"VisDrone2019-DET-{split}"/"images").glob("*.jpg"))
    list_path = DIRS["output"]/f"visdrone_{split}_files.txt"
    list_path.write_text("\n".join(files))
    h = hashlib.sha256("\n".join(files).encode()).hexdigest()
    hashes[split] = {"n_files": len(files), "sha256": h}
    print(f"{split}: n={len(files)}  sha256={h[:16]}...")

with open(DIRS["output"]/"visdrone_split_hashes.json","w") as f:
    json.dump(hashes, f, indent=2)
print("Split FROZEN ->", DIRS["output"]/"visdrone_split_hashes.json")


## Leakage audit #1 — filename overlap + exact-duplicate (MD5) scan
Protocol §1(iii) near-duplicate CLIP scan runs later, after the gallery is built (notebook 04). Here: hard guarantees only.

In [ ]:
def md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p,"rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

names = {s: set((DIRS["output"]/f"visdrone_{s}_files.txt").read_text().split()) for s in ["train","val"]}
overlap = names["train"] & names["val"]
print("Filename overlap train∩val:", len(overlap))

md5s = {}
for split in ["train","val"]:
    for p in tqdm(sorted((DATA/f"VisDrone2019-DET-{split}"/"images").glob("*.jpg")), desc=f"md5-{split}"):
        md5s.setdefault(md5(p), []).append(f"{split}/{p.name}")
cross = [v for v in md5s.values() if len(v)>1 and len({x.split('/')[0] for x in v})>1]
print("Exact duplicates across splits:", len(cross))
assert not overlap and not cross, "LEAKAGE: split overlap detected!"
print("Leakage audit #1: CLEAN")


## Data card (protocol §4.1) — saved to `output/datacard_visdrone.csv`

In [ ]:
import pandas as pd

card = pd.DataFrame([
    ["Dataset", "VisDrone2019-DET"],
    ["Role in study", "Primary benchmark (protocol v1.0)"],
    ["Train images", hashes["train"]["n_files"]],
    ["Val (eval) images", hashes["val"]["n_files"]],
    ["Classes", len(CONFIG["CLASSES"])],
    ["Train instances", sum(counts["train"].values())],
    ["Val instances", sum(counts["val"].values())],
    ["Class imbalance (max/min, train)",
     round(max(counts['train'].values())/max(1,min(counts['train'].values())),1)],
    ["Image size (training)", "640 (protocol v1.0)"],
    ["License / access", "VisDrone (AISKYEYE, academic use); mirrors: ultralytics assets"],
    ["Preprocessing", "score==1, cat 1-10 kept, boxes clipped+normalized (this notebook)"],
    ["Split hash (train)", hashes["train"]["sha256"][:32]+"..."],
    ["Split hash (val)", hashes["val"]["sha256"][:32]+"..."],
], columns=["field","value"])
card.to_csv(DIRS["output"]/"datacard_visdrone.csv", index=False)
card


## YOLO data YAML (consumed by all later notebooks)
Written once here; later notebooks **read** it, never redefine paths.

**Next:** `02_data_coco10.ipynb` (COCO-standard 10% split + unlabeled gallery pool), then `03_data_exdark.ipynb`.

In [ ]:
yaml_path = DIRS["dataset"] / "visdrone_raodpp.yaml"
yaml_path.write_text(f"""# RAOD++ VisDrone2019-DET (protocol v1.0) — generated by 01_data_visdrone.ipynb
path: {DATA}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
names:
{chr(10).join(f'  {i}: {n}' for i,n in enumerate(CONFIG['CLASSES']))}
""")
print("Wrote:", yaml_path)
print(yaml_path.read_text())
